# Mask R-CNN Pipeline: Crop → SAM → Train (Kaggle)

Complete workflow — generates crops from YOLO labels, runs SAM for masks, then trains Mask R-CNN.

## Kaggle Setup

1. **Upload YOLO dataset** as Kaggle Dataset: upload `data/yolo/` folder
   - Must contain: `images/{train,val}/`, `labels/{train,val}/`
   - Name it e.g. `mp-yolo-data`

2. **Upload SAM checkpoint** as separate Kaggle Dataset:
   - Upload `sam_vit_h_4b8939.pth` (~2.4 GB)
   - Name it e.g. `sam-vit-h-checkpoint`
   - *Or* the notebook will download it from Meta (slower)

3. **Add both datasets** to notebook via sidebar → Add Data

4. **Enable GPU** (Settings → Accelerator → GPU T4 x2)

5. **Enable Internet** (Settings → Internet → On)

## Where Models Are Saved

```
/kaggle/working/
├── experiments/
│   ├── maskrcnn_crops_best.pth     ← DOWNLOAD THIS
│   └── maskrcnn_crops_latest.pth
├── crops/                           ← YOLO crops (intermediate)
├── crops_sam/                       ← SAM-annotated crops
└── yolo/                            ← local copy of YOLO data
```

After training: **Save Version** → download from Output tab.

## 1. Install & Import

In [ ]:
!pip install -q torch torchvision pycocotools opencv-python-headless
!pip install -q albumentations ultralytics timm tqdm matplotlib
!pip install -q segment-anything

In [ ]:
import json, cv2, shutil, os, random
import numpy as np, torch
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt

from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from segment_anything import SamPredictor, sam_model_registry

print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Embedded Functions

In [ ]:
NUM_CLASSES = 4
CLASS_NAMES = ['background', 'fiber', 'film', 'fragment']
YOLO_TO_MASKRCNN = {0: 1, 1: 2, 2: 3}


def prepare_from_yolo_labels(yolo_dir, output_dir, padding=20, splits=('train','val')):
    yolo_path = Path(yolo_dir); out_path = Path(output_dir)
    (out_path / 'images').mkdir(parents=True, exist_ok=True)
    for cn in CLASS_NAMES[1:]: (out_path / cn).mkdir(parents=True, exist_ok=True)

    annotations = {}; crop_count = 0; class_counts = {0:0, 1:0, 2:0}
    for split in splits:
        img_dir = yolo_path / 'images' / split
        lbl_dir = yolo_path / 'labels' / split
        if not img_dir.exists(): print(f'Skip {split}'); continue
        files = sorted(list(img_dir.glob('*.png')) + list(img_dir.glob('*.jpg')))
        print(f'[{split}] {len(files)} images')
        for img_path in tqdm(files, desc=f'Cropping {split}'):
            image = cv2.imread(str(img_path))
            if image is None: continue
            h, w = image.shape[:2]
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists(): continue
            with open(lbl_path) as f: lines = [l.strip() for l in f if l.strip()]
            for line in lines:
                parts = line.split()
                if len(parts) < 5: continue
                cls_id = int(float(parts[0]))
                cx_n, cy_n, bw_n, bh_n = map(float, parts[1:5])
                cx, cy, bw, bh = cx_n*w, cy_n*h, bw_n*w, bh_n*h
                dx1, dy1 = int(cx-bw/2), int(cy-bh/2)
                dx2, dy2 = int(cx+bw/2), int(cy+bh/2)
                x1, y1 = max(0,dx1-padding), max(0,dy1-padding)
                x2, y2 = min(w,dx2+padding), min(h,dy2+padding)
                crop = image[y1:y2, x1:x2]
                if crop.size == 0 or crop.shape[0] < 10 or crop.shape[1] < 10: continue
                cn = CLASS_NAMES[YOLO_TO_MASKRCNN[cls_id]]
                fname = f'{img_path.stem}_gt{crop_count:04d}_{cn}.png'
                cv2.imwrite(str(out_path / 'images' / fname), crop)
                cv2.imwrite(str(out_path / cn / fname), crop)
                annotations[fname] = {'source_image': img_path.name, 'split': split,
                    'class_id': cls_id, 'class_name': cn, 'yolo_confidence': 1.0,
                    'rel_box': [dx1-x1, dy1-y1, dx2-x1, dy2-y1],
                    'crop_size': [crop.shape[1], crop.shape[0]]}
                class_counts[cls_id] += 1; crop_count += 1

    with open(out_path / 'annotations.json', 'w') as f: json.dump(annotations, f, indent=2)
    print(f'\nCrops: {crop_count} (fiber={class_counts[0]} film={class_counts[1]} fragment={class_counts[2]})')


class CropDataset(Dataset):
    CLASS_NAME_TO_YOLO_ID = {'fiber': 0, 'film': 1, 'fragment': 2}

    def __init__(self, crops_dir, transforms=None):
        self.crops_dir = Path(crops_dir); self.transforms = transforms
        ann_file = self.crops_dir / 'annotations.json'
        has_flat = (self.crops_dir / 'images').is_dir()
        has_cls = any((self.crops_dir / c).is_dir() for c in self.CLASS_NAME_TO_YOLO_ID)
        if ann_file.exists():
            with open(ann_file) as f: self.annotations = json.load(f)
            self.images_dir = (self.crops_dir / 'images') if has_flat else None
        elif has_cls:
            self.annotations = {}; self.images_dir = None
            for cn, ci in self.CLASS_NAME_TO_YOLO_ID.items():
                cd = self.crops_dir / cn
                if not cd.is_dir(): continue
                for f in sorted(cd.glob('*.png')):
                    img = cv2.imread(str(f))
                    if img is None: continue
                    h, w = img.shape[:2]
                    self.annotations[f.name] = {'source_image':'','class_id':ci,'class_name':cn,
                        'yolo_confidence':1.0,'rel_box':[0,0,w,h],'crop_size':[w,h]}
        else: raise FileNotFoundError(f'No data in {crops_dir}')
        self.samples = list(self.annotations.keys())
        print(f'CropDataset: {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]; ann = self.annotations[name]
        img_path = (self.images_dir / name) if self.images_dir else (self.crops_dir / ann.get('class_name','') / name)
        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]
        mask = None; masks_dir = self.crops_dir / 'masks'
        mf = ann.get('mask_file')
        if mf and (masks_dir/mf).exists():
            raw = cv2.imread(str(masks_dir/mf), cv2.IMREAD_GRAYSCALE)
            if raw is not None: mask = (raw>127).astype(np.uint8)
        if mask is None:
            dm = masks_dir / name.replace('.png','_mask.png')
            if dm.exists():
                raw = cv2.imread(str(dm), cv2.IMREAD_GRAYSCALE)
                if raw is not None: mask = (raw>127).astype(np.uint8)
        if mask is None:
            mask = np.zeros((h,w), np.uint8)
            rb = ann.get('rel_box')
            cx,cy = ((rb[0]+rb[2])//2,(rb[1]+rb[3])//2) if rb else (w//2,h//2)
            ax,ay = ((rb[2]-rb[0])//2,(rb[3]-rb[1])//2) if rb else (int(w*0.4),int(h*0.4))
            if ax>0 and ay>0: cv2.ellipse(mask,(cx,cy),(ax,ay),0,0,360,1,-1)
        if mask.shape[:2]!=(h,w): mask = cv2.resize(mask,(w,h),interpolation=cv2.INTER_NEAREST)
        ys,xs = np.where(mask>0)
        box = [xs.min(),ys.min(),xs.max(),ys.max()] if len(xs)>0 else [min(h,w)//10]*2+[w-min(h,w)//10,h-min(h,w)//10]
        class_id = YOLO_TO_MASKRCNN[ann['class_id']]
        boxes = np.array([box], dtype=np.float32)
        labels = np.array([class_id], dtype=np.int64)
        masks = np.array([mask], dtype=np.uint8)
        if self.transforms:
            t = self.transforms(image=image, bboxes=boxes.tolist(), masks=list(masks), class_labels=labels.tolist())
            image = t['image']
            if len(t['bboxes'])>0:
                boxes = np.array(t['bboxes'], np.float32)
                labels = np.array(t['class_labels'], np.int64)
                masks = np.array(t['masks'], np.uint8)
        else: image = torch.from_numpy(image.transpose(2,0,1)).float()/255.0
        return image, {'boxes':torch.as_tensor(boxes,dtype=torch.float32),
            'labels':torch.as_tensor(labels,dtype=torch.int64),
            'masks':torch.as_tensor(masks,dtype=torch.uint8),
            'image_id':torch.tensor([idx]),
            'area':torch.as_tensor([(b[2]-b[0])*(b[3]-b[1]) for b in boxes],dtype=torch.float32),
            'iscrowd':torch.zeros(len(boxes),dtype=torch.int64)}


def get_transforms(train=True, img_size=128):
    if train:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            A.GaussNoise(var_limit=(10.,50.), p=0.3),
            A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2(),
        ], bbox_params=A.BboxParams('pascal_voc', label_fields=['class_labels'], min_visibility=0.3))
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2(),
    ], bbox_params=A.BboxParams('pascal_voc', label_fields=['class_labels'], min_visibility=0.3))

def collate_fn(batch): return tuple(zip(*batch))

def get_model(num_classes, pretrained=True):
    w = MaskRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained else None
    model = maskrcnn_resnet50_fpn(weights=w)
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, num_classes)
    inf_m = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(inf_m, 256, num_classes)
    return model

print('All functions defined')

## 3. Configuration & Data Copy

In [ ]:
# >>> CHANGE THESE <<<
YOLO_DATASET_NAME = 'mp-yolo-data'           # YOLO images + labels
SAM_CKPT_DATASET  = 'sam-vit-h-checkpoint'   # SAM checkpoint (or None to download)

# Kaggle paths
YOLO_INPUT = Path(f'/kaggle/input/{YOLO_DATASET_NAME}')
SAM_INPUT  = Path(f'/kaggle/input/{SAM_CKPT_DATASET}') if SAM_CKPT_DATASET else None

# Local working paths
YOLO_DIR       = Path('/kaggle/working/yolo')
CROPS_DIR      = Path('/kaggle/working/crops')
SAM_OUTPUT_DIR = Path('/kaggle/working/crops_sam')
SAM_CHECKPOINT = Path('/kaggle/working/sam_vit_h_4b8939.pth')
SAVE_DIR       = Path('/kaggle/working/experiments')

assert YOLO_INPUT.exists(), f'YOLO dataset not found: {YOLO_INPUT}\nAvailable: {os.listdir("/kaggle/input/")}'

# Auto-detect YOLO structure
if (YOLO_INPUT / 'images').exists():
    YOLO_SRC = YOLO_INPUT
elif (YOLO_INPUT / 'yolo' / 'images').exists():
    YOLO_SRC = YOLO_INPUT / 'yolo'
else:
    found = list(YOLO_INPUT.rglob('images/train'))
    assert found, f'No images/train found under {YOLO_INPUT}'
    YOLO_SRC = found[0].parent.parent

# Copy YOLO data
if not YOLO_DIR.exists():
    print(f'Copying YOLO data {YOLO_SRC} -> {YOLO_DIR}...')
    shutil.copytree(str(YOLO_SRC), str(YOLO_DIR))
    print('Done.')
else:
    print(f'YOLO data already copied: {YOLO_DIR}')

# SAM checkpoint
if not SAM_CHECKPOINT.exists():
    if SAM_INPUT:
        ckpt_candidates = list(SAM_INPUT.rglob('*.pth'))
        if ckpt_candidates:
            print(f'Copying SAM checkpoint...')
            shutil.copy2(str(ckpt_candidates[0]), str(SAM_CHECKPOINT))
        else:
            print('No .pth in SAM dataset, downloading...')
            import urllib.request
            urllib.request.urlretrieve(
                'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
                str(SAM_CHECKPOINT))
    else:
        print('Downloading SAM checkpoint (~2.4 GB)...')
        import urllib.request
        urllib.request.urlretrieve(
            'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
            str(SAM_CHECKPOINT))
    print(f'SAM checkpoint ready: {SAM_CHECKPOINT}')

for d in [CROPS_DIR, SAM_OUTPUT_DIR, SAVE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Config
SAM_MODEL_TYPE = 'vit_h'
CROP_PADDING = 20
MASKRCNN_EPOCHS = 50
MASKRCNN_BATCH_SIZE = 8
MASKRCNN_LR = 0.001
CROP_SIZE = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'\nDevice: {DEVICE}')
print(f'YOLO: {YOLO_DIR}')
print(f'SAM:  {SAM_CHECKPOINT}')
print(f'Save: {SAVE_DIR}')

## 4. Explore YOLO Dataset

In [ ]:
yolo_path = Path(YOLO_DIR)
for split in ['train','val']:
    id_ = yolo_path / 'images' / split
    ld_ = yolo_path / 'labels' / split
    if id_.exists():
        imgs = list(id_.glob('*.png')) + list(id_.glob('*.jpg'))
        lbls = list(ld_.glob('*.txt')) if ld_.exists() else []
        print(f'[{split}] {len(imgs)} images, {len(lbls)} labels')

class_counts = defaultdict(int); total = 0
for split in ['train','val']:
    ld_ = yolo_path / 'labels' / split
    if not ld_.exists(): continue
    for lf in ld_.glob('*.txt'):
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts)>=5:
                    cid = int(float(parts[0]))
                    class_counts[CLASS_NAMES[YOLO_TO_MASKRCNN.get(cid,0)]] += 1; total += 1
print(f'\nTotal objects: {total}')
for c,n in sorted(class_counts.items()): print(f'  {c}: {n}')

## 5. Generate Crops from YOLO Labels

In [ ]:
prepare_from_yolo_labels(
    yolo_dir=str(YOLO_DIR),
    output_dir=str(CROPS_DIR),
    padding=CROP_PADDING
)

In [ ]:
# Verify crops
with open(CROPS_DIR / 'annotations.json') as f:
    crop_ann = json.load(f)
print(f'Total crops: {len(crop_ann)}')
by_cls = defaultdict(int)
for a in crop_ann.values(): by_cls[a['class_name']] += 1
for c,n in sorted(by_cls.items()): print(f'  {c}: {n}')

## 6. Initialize SAM & Generate Masks

In [ ]:
print(f'Loading SAM ({SAM_MODEL_TYPE})...')
sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=str(SAM_CHECKPOINT))
sam.to(DEVICE); sam.eval()
sam_predictor = SamPredictor(sam)
print(f'SAM loaded on {DEVICE}')

In [ ]:
crops_path = Path(CROPS_DIR)
sam_out = Path(SAM_OUTPUT_DIR)
(sam_out / 'images').mkdir(parents=True, exist_ok=True)
(sam_out / 'masks').mkdir(parents=True, exist_ok=True)

with open(crops_path / 'annotations.json') as f:
    crop_ann = json.load(f)

print(f'Generating SAM masks for {len(crop_ann)} crops...')
sam_ann = {}; ok = 0; fail = 0; scores_list = []

for name, ann in tqdm(crop_ann.items(), desc='SAM masks'):
    ip = crops_path / 'images' / name
    if not ip.exists(): fail += 1; continue
    image = cv2.imread(str(ip))
    if image is None: fail += 1; continue
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    try:
        sam_predictor.set_image(rgb)
        pt = np.array([[w//2, h//2]]); pl = np.array([1])
        rb = ann.get('rel_box')
        if rb:
            masks, scores, _ = sam_predictor.predict(point_coords=pt, point_labels=pl,
                                                     box=np.array(rb), multimask_output=True)
        else:
            masks, scores, _ = sam_predictor.predict(point_coords=pt, point_labels=pl,
                                                     multimask_output=True)
        best = np.argmax(scores)
        best_mask = masks[best].astype(np.uint8)
        best_score = float(scores[best])
        mf = name.replace('.png','_mask.png').replace('.jpg','_mask.png')
        cv2.imwrite(str(sam_out / 'masks' / mf), best_mask * 255)
        cv2.imwrite(str(sam_out / 'images' / name), image)
        new_ann = ann.copy()
        new_ann['mask_file'] = mf; new_ann['sam_score'] = best_score
        sam_ann[name] = new_ann
        scores_list.append(best_score); ok += 1
    except Exception as e:
        print(f'  Error {name}: {e}'); fail += 1

with open(sam_out / 'annotations.json', 'w') as f:
    json.dump(sam_ann, f, indent=2)

print(f'\nSAM done: {ok} success, {fail} failed')
if scores_list:
    print(f'Score: mean={np.mean(scores_list):.3f} median={np.median(scores_list):.3f}')

## 7. SAM Quality Check

In [ ]:
with open(SAM_OUTPUT_DIR / 'annotations.json') as f:
    sa = json.load(f)

scores = [a.get('sam_score',0) for a in sa.values()]
print(f'Masks: {len(scores)}')
print(f'Scores: mean={np.mean(scores):.3f} min={np.min(scores):.3f} max={np.max(scores):.3f}')
print(f'  High (>=0.9): {sum(1 for s in scores if s>=0.9)}')
print(f'  Low  (<0.7):  {sum(1 for s in scores if s<0.7)}')

by_cls = defaultdict(list)
for a in sa.values(): by_cls[a['class_name']].append(a.get('sam_score',0))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(scores, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(0.7, color='red', ls='--'); axes[0].set_xlabel('Score'); axes[0].set_title('SAM Score Distribution')
axes[1].boxplot([by_cls[c] for c in ['fiber','film','fragment']], labels=['fiber','film','fragment'])
axes[1].set_ylabel('Score'); axes[1].set_title('Score by Class')
plt.tight_layout(); plt.show()

# Visual samples
sample_keys = random.sample(list(sa.keys()), min(6, len(sa)))
fig, axes = plt.subplots(2, 6, figsize=(24, 8))
for j, name in enumerate(sample_keys):
    ann = sa[name]
    img = cv2.cvtColor(cv2.imread(str(SAM_OUTPUT_DIR/'images'/name)), cv2.COLOR_BGR2RGB)
    mf = ann.get('mask_file', name.replace('.png','_mask.png'))
    axes[0,j].imshow(img); axes[0,j].set_title(ann['class_name'], fontsize=9); axes[0,j].axis('off')
    mp = SAM_OUTPUT_DIR / 'masks' / mf
    if mp.exists():
        m = (cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)>127).astype(np.uint8)
        o = img.copy(); o[m==1] = (o[m==1]*0.5 + np.array([0,255,0])*0.5).astype(np.uint8)
        axes[1,j].imshow(o)
    axes[1,j].set_title(f'score={ann.get("sam_score",0):.3f}', fontsize=9); axes[1,j].axis('off')
plt.tight_layout(); plt.show()

## 8. Free SAM from GPU Memory

In [ ]:
del sam, sam_predictor
torch.cuda.empty_cache()
import gc; gc.collect()
print('SAM unloaded, GPU memory freed')

## 9. Build Dataset & Model

In [ ]:
train_tf = get_transforms(train=True, img_size=CROP_SIZE)
train_dataset = CropDataset(str(SAM_OUTPUT_DIR), transforms=train_tf)

sample_img, sample_tgt = train_dataset[0]
print(f'Image: {sample_img.shape}, Label: {CLASS_NAMES[sample_tgt["labels"][0].item()]}')

model = get_model(NUM_CLASSES, pretrained=True).to(DEVICE)
print(f'Mask R-CNN: {sum(p.numel() for p in model.parameters()):,} params')

## 10. Train Mask R-CNN

In [ ]:
train_loader = DataLoader(train_dataset, MASKRCNN_BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=MASKRCNN_LR, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MASKRCNN_EPOCHS,
                                                        eta_min=MASKRCNN_LR*0.01)

best_loss = float('inf')
history = {'epoch':[], 'loss':[], 'lr':[], 'loss_classifier':[], 'loss_box_reg':[],
           'loss_mask':[], 'loss_objectness':[], 'loss_rpn_box_reg':[]}

# Auto-backup every 10 epochs
BACKUP_EVERY = 10
BACKUP_ROOT  = Path('/kaggle/working/backups')

print(f'Training {MASKRCNN_EPOCHS} epochs, batch={MASKRCNN_BATCH_SIZE}, LR={MASKRCNN_LR}')
print(f'Auto-backup: every {BACKUP_EVERY} epochs → {BACKUP_ROOT}')

for epoch in range(MASKRCNN_EPOCHS):
    model.train(); epoch_loss = 0.; epoch_comp = defaultdict(float); bc = 0
    pbar = tqdm(train_loader, desc=f'Ep {epoch+1}/{MASKRCNN_EPOCHS}')
    for images, targets in pbar:
        images = [i.to(DEVICE) for i in images]
        targets = [{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
        if not all(len(t['boxes'])>0 for t in targets): continue
        ld = model(images, targets); losses = sum(ld.values())
        optimizer.zero_grad(); losses.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0); optimizer.step()
        epoch_loss += losses.item(); bc += 1
        for k,v in ld.items(): epoch_comp[k] += v.item()
        pbar.set_postfix(loss=f'{losses.item():.4f}')

    scheduler.step()
    avg = epoch_loss / max(bc,1)
    history['epoch'].append(epoch+1); history['loss'].append(avg)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    for k in ['loss_classifier','loss_box_reg','loss_mask','loss_objectness','loss_rpn_box_reg']:
        history[k].append(epoch_comp.get(k,0)/max(bc,1))

    ckpt = {'epoch':epoch+1, 'model_state_dict':model.state_dict(),
            'optimizer_state_dict':optimizer.state_dict(), 'loss':avg, 'history':history}
    torch.save(ckpt, str(SAVE_DIR / 'maskrcnn_crops_latest.pth'))
    tag = ''
    if avg < best_loss:
        best_loss = avg
        torch.save(ckpt, str(SAVE_DIR / 'maskrcnn_crops_best.pth'))
        tag = ' * best'

    # Auto-backup
    if (epoch + 1) % BACKUP_EVERY == 0:
        backup_dir = BACKUP_ROOT / f'epoch_{epoch+1:04d}'
        backup_dir.mkdir(parents=True, exist_ok=True)
        for name in ('maskrcnn_crops_best.pth', 'maskrcnn_crops_latest.pth'):
            src = SAVE_DIR / name
            if src.exists():
                shutil.copy2(str(src), str(backup_dir / name))
        print(f'  [AUTO-BACKUP] epoch {epoch+1} → {backup_dir}')

    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f'  Ep {epoch+1}/{MASKRCNN_EPOCHS} loss={avg:.4f}{tag}')

print(f'\nDone — best loss {best_loss:.4f}')


## 11. Training Curves

In [ ]:
# ==============================================================================
# 11. TRAINING CURVES (self-contained)
# ==============================================================================

import torch, matplotlib.pyplot as plt
from pathlib import Path

SAVE_DIR    = Path('/kaggle/working/experiments')
BACKUP_ROOT = Path('/kaggle/working/backups')

def find_best_checkpoint(save_dir, backup_root):
    primary = save_dir / 'maskrcnn_crops_best.pth'
    if primary.exists():
        return str(primary)
    backups = sorted(backup_root.glob('epoch_*/maskrcnn_crops_best.pth')) if backup_root.exists() else []
    if backups:
        return str(backups[-1])
    candidates = sorted(Path('/kaggle/working').rglob('maskrcnn_crops_best.pth'))
    if candidates:
        return str(candidates[-1])
    raise FileNotFoundError('maskrcnn_crops_best.pth not found.')

ckpt_path = find_best_checkpoint(SAVE_DIR, BACKUP_ROOT)
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
history = ckpt['history']
print(f'Loaded history from {ckpt_path} (epoch {ckpt["epoch"]})')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ep = history['epoch']

axes[0,0].plot(ep, history['loss'], 'b-', lw=2); axes[0,0].set_title('Total Loss'); axes[0,0].grid(True,alpha=0.3)
for k,c,l in [('loss_classifier','r','Classif'),('loss_box_reg','g','Box'),
              ('loss_mask','b','Mask'),('loss_objectness','m','Obj'),('loss_rpn_box_reg','c','RPN')]:
    if history.get(k):
        axes[0,1].plot(ep, history[k], color=c, label=l, lw=1.5)
axes[0,1].set_title('Components'); axes[0,1].legend(fontsize=8); axes[0,1].grid(True,alpha=0.3)
axes[1,0].plot(ep, history['lr'], 'g-', lw=2); axes[1,0].set_title('LR'); axes[1,0].grid(True,alpha=0.3)
if history.get('loss_mask'):
    axes[1,1].plot(ep, history['loss_mask'], 'b-', lw=2); axes[1,1].set_title('Mask Loss'); axes[1,1].grid(True,alpha=0.3)
plt.tight_layout(); plt.show()


## 12. Visualise Predictions

In [ ]:
# ==============================================================================
# 12. VISUALISE PREDICTIONS (self-contained)
# ==============================================================================

import json, random, torch, cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from torchvision import transforms as T
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

NUM_CLASSES = 4
CLASS_NAMES = ['background', 'fiber', 'film', 'fragment']
CROP_SIZE   = 128
SAVE_DIR    = Path('/kaggle/working/experiments')
SAM_OUTPUT_DIR = Path('/kaggle/working/crops_sam')
BACKUP_ROOT = Path('/kaggle/working/backups')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def find_best_checkpoint(save_dir, backup_root):
    primary = save_dir / 'maskrcnn_crops_best.pth'
    if primary.exists():
        return str(primary)
    backups = sorted(backup_root.glob('epoch_*/maskrcnn_crops_best.pth')) if backup_root.exists() else []
    if backups:
        return str(backups[-1])
    candidates = sorted(Path('/kaggle/working').rglob('maskrcnn_crops_best.pth'))
    if candidates:
        return str(candidates[-1])
    raise FileNotFoundError('maskrcnn_crops_best.pth not found.')

def get_model(num_classes):
    m = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, num_classes)
    inf_m = m.roi_heads.mask_predictor.conv5_mask.in_channels
    m.roi_heads.mask_predictor = MaskRCNNPredictor(inf_m, 256, num_classes)
    return m

ckpt_path = find_best_checkpoint(SAVE_DIR, BACKUP_ROOT)
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model = get_model(NUM_CLASSES).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict']); model.eval()
print(f'Loaded best model (epoch {ckpt["epoch"]}, loss {ckpt["loss"]:.4f})')

inf_tf = T.Compose([T.ToPILImage(), T.Resize((CROP_SIZE,CROP_SIZE)),
                     T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

with open(SAM_OUTPUT_DIR / 'annotations.json') as f: ta = json.load(f)
samples = random.sample(list(ta.keys()), min(8, len(ta)))
class_colors = {1:(255,50,50), 2:(50,255,50), 3:(50,50,255)}

fig, axes = plt.subplots(len(samples), 4, figsize=(20, 4*len(samples)))
if len(samples)==1: axes = axes.reshape(1,-1)
for col, title in enumerate(['Original','SAM GT','Pred Mask','Overlay']):
    axes[0,col].set_title(title, fontsize=12, fontweight='bold')

for i, name in enumerate(samples):
    ann = ta[name]
    img = cv2.cvtColor(cv2.imread(str(SAM_OUTPUT_DIR/'images'/name)), cv2.COLOR_BGR2RGB)
    axes[i,0].imshow(img); axes[i,0].set_ylabel(ann['class_name'], fontsize=10, rotation=0, labelpad=50); axes[i,0].axis('off')

    mf = ann.get('mask_file', name.replace('.png','_mask.png'))
    gp = SAM_OUTPUT_DIR / 'masks' / mf
    if gp.exists():
        gb = (cv2.imread(str(gp),cv2.IMREAD_GRAYSCALE)>127).astype(np.uint8)
        go = img.copy(); go[gb==1] = (go[gb==1]*0.5+np.array([0,255,0])*0.5).astype(np.uint8)
        axes[i,1].imshow(go)
    axes[i,1].axis('off')

    inp = inf_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): pred = model(inp)[0]
    keep = pred['scores'] > 0.3
    if keep.sum() > 0:
        bm = pred['masks'][keep][0,0].cpu().numpy()
        bl = pred['labels'][keep][0].item()
        bs = pred['scores'][keep][0].item()
        axes[i,2].imshow(bm>0.5, cmap='gray')
        axes[i,2].set_title(f'{CLASS_NAMES[bl]} ({bs:.2f})', fontsize=9); axes[i,2].axis('off')
        ir = cv2.resize(img,(CROP_SIZE,CROP_SIZE))
        pb = (bm>0.5).astype(np.uint8)
        c = np.array(class_colors.get(bl,(255,255,0)))/255.
        ov = ir.copy().astype(float)/255; ov[pb==1] = ov[pb==1]*0.5+c*0.5
        axes[i,3].imshow(ov); axes[i,3].axis('off')
    else:
        axes[i,2].text(0.5,0.5,'No det',ha='center',va='center'); axes[i,2].axis('off')
        axes[i,3].text(0.5,0.5,'No det',ha='center',va='center'); axes[i,3].axis('off')

plt.tight_layout(); plt.show()


## 13. Save & Download

Models are in `/kaggle/working/experiments/`.
Click **Save Version** → download from **Output** tab.

In [ ]:
# ==============================================================================
# 13. SAVE & DOWNLOAD (self-contained)
# ==============================================================================

import shutil
from pathlib import Path

SAVE_DIR    = Path('/kaggle/working/experiments')
BACKUP_ROOT = Path('/kaggle/working/backups')

def find_best_checkpoint(save_dir, backup_root):
    primary = save_dir / 'maskrcnn_crops_best.pth'
    if primary.exists():
        return str(primary)
    backups = sorted(backup_root.glob('epoch_*/maskrcnn_crops_best.pth')) if backup_root.exists() else []
    if backups:
        return str(backups[-1])
    candidates = sorted(Path('/kaggle/working').rglob('maskrcnn_crops_best.pth'))
    if candidates:
        return str(candidates[-1])
    raise FileNotFoundError('maskrcnn_crops_best.pth not found.')

ckpt_path = find_best_checkpoint(SAVE_DIR, BACKUP_ROOT)
shutil.copy2(ckpt_path, '/kaggle/working/maskrcnn_crops_best.pth')

for f in SAVE_DIR.glob('*.pth'):
    print(f'{f.name}: {f.stat().st_size/1e6:.1f} MB')

print(f'\nCopied from: {ckpt_path}')
print('Save Version -> download from Output tab')
